In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("house_prices_srilanka.csv")

print(df.shape)
df.head()

(20000, 14)


,district,area,perch,bedrooms,bathrooms,kitchen_area_sqft,parking_spots,has_garden,has_ac,water_supply,electricity,floors,year_built,price_lkr
0,Polonnaruwa,Polonnaruwa Central,14,5,4,112,2,False,False,Pipe-borne,Single phase,1,2022,16236332
1,Matale,Matale Central,26,7,5,250,2,True,False,Both,Single phase,3,1987,33907509
2,Mullaitivu,Mullaitivu Central,7,3,2,77,2,True,True,Well,Single phase,1,1988,5954312
3,Anuradhapura,New Town,2,2,2,39,1,True,True,Both,Three phase,2,2024,5177534
4,Batticaloa,Batticaloa Town,20,5,4,117,1,True,True,Well,Single phase,1,1995,20182409


In [5]:
df.isnull().sum()

district             0
area                 0
perch                0
bedrooms             0
bathrooms            0
kitchen_area_sqft    0
parking_spots        0
has_garden           0
has_ac               0
water_supply         0
electricity          0
floors               0
year_built           0
price_lkr            0
dtype: int64

In [7]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Removed duplicates:", before - after)

print(df.shape)

Removed duplicates: 0
(20000, 14)


In [9]:
df = df[
    (df["price_lkr"] > 1000000) &
    (df["price_lkr"] < 200000000)
]

print(df.shape)

(20000, 14)


In [11]:
CURRENT_YEAR = 2026

df["house_age"] = CURRENT_YEAR - df["year_built"]

df.head()

,district,area,perch,bedrooms,bathrooms,kitchen_area_sqft,parking_spots,has_garden,has_ac,water_supply,electricity,floors,year_built,price_lkr,house_age
0,Polonnaruwa,Polonnaruwa Central,14,5,4,112,2,False,False,Pipe-borne,Single phase,1,2022,16236332,4
1,Matale,Matale Central,26,7,5,250,2,True,False,Both,Single phase,3,1987,33907509,39
2,Mullaitivu,Mullaitivu Central,7,3,2,77,2,True,True,Well,Single phase,1,1988,5954312,38
3,Anuradhapura,New Town,2,2,2,39,1,True,True,Both,Three phase,2,2024,5177534,2
4,Batticaloa,Batticaloa Town,20,5,4,117,1,True,True,Well,Single phase,1,1995,20182409,31


In [13]:
FEATURES = [
    "district",
    "area",
    "perch",
    "bedrooms",
    "bathrooms",
    "kitchen_area_sqft",
    "parking_spots",
    "has_garden",
    "has_ac",
    "water_supply",
    "electricity",
    "floors",
    "house_age"
]

TARGET = "price_lkr"

In [15]:
X = df[FEATURES]

y = df[TARGET]

In [17]:
categorical_cols = [
    "district",
    "area",
    "water_supply",
    "electricity"
]

numerical_cols = [
    "perch",
    "bedrooms",
    "bathrooms",
    "kitchen_area_sqft",
    "parking_spots",
    "floors",
    "house_age"
]

boolean_cols = [
    "has_garden",
    "has_ac"
]

In [19]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            StandardScaler(),
            numerical_cols
        ),

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),

        (
            "bool",
            "passthrough",
            boolean_cols
        )
    ]
)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42
)

print(X_train.shape)

print(X_test.shape)

(16000, 13)
(4000, 13)


In [ ]:
model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "regressor",

        RandomForestRegressor(

            n_estimators=50,

            max_depth=10,

            random_state=42,

            n_jobs=-1
        )
    )
])

In [25]:
model.fit(
    X_train,
    y_train
)

print("Model trained successfully!")

Model trained successfully!


In [27]:
predictions = model.predict(X_test)

In [29]:
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

print("MAE :", mae)

print("RMSE:", rmse)

print("R2  :", r2)

MAE : 1776161.3804849864
RMSE: 2734274.295510203
R2  : 0.9579732765177454


In [31]:
df.to_csv(

    "cleaned_house_data.csv",

    index=False
)

print("Cleaned dataset saved!")

Cleaned dataset saved!


In [33]:
joblib.dump(

    model,

    "house_price_model.pkl"
)

print("Model saved successfully!")

Model saved successfully!


In [35]:
sample_house = pd.DataFrame([{

    "district": "Colombo",

    "area": "Colombo 03",

    "perch": 10,

    "bedrooms": 3,

    "bathrooms": 2,

    "kitchen_area_sqft": 120,

    "parking_spots": 1,

    "has_garden": True,

    "has_ac": True,

    "water_supply": "Pipe-borne",

    "electricity": "Three phase",

    "floors": 2,

    "house_age": 5
}])

predicted_price = model.predict(
    sample_house
)

print(
    "Predicted Price (LKR):",

    round(predicted_price[0], 2)
)

Predicted Price (LKR): 38639118.17
